# Tutorial 2: Training

This tutorial covers training the deepSCENIC model.

## Learning Objectives

- Understand the deepSCENIC architecture
- Configure training parameters
- Train the model with optional pretraining and finetuning stages
- Monitor training progress
- Save and load models

## Setup

In [4]:
import deepscenic as ds
import torch

print("deepSCENIC version:", ds.__version__)
# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

deepSCENIC version: 0.1.0
Using device: cuda


## Load Preprocessed Data

In [5]:
# Load preprocessed MuData
import os
data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "dir not found"

mdata = ds.read(data_dir + "preprocessed_data.h5mu")
mdata

MuData object with n_obs × n_vars = 936 × 292241
  obs:	'split'
  uns:	'r2g'
  2 modalities
    rna:	936 x 10421
      var:	'n_cells', 'is_tf', 'chromosome', 'tss', 'split'
      uns:	'log1p', 'tf_order'
      layers:	'log_norm'
    atac:	936 x 281820
      var:	'chromosome', 'start', 'end', 'split'

## Understanding the Architecture

deepSCENIC consists of:

1. **TF2rNet**: Predicts TF binding from DNA sequence (Enformer + MotifNet)
2. **PPI Network**: Models post-transcriptional TF regulation (GAT)
3. **VAE**: Transforms TF expression to regulatory activity
4. **GRN Matrices**: E1 (TF->region) and E2 (region->gene)

```
TF expression
  | PPI modulation
  | InferenceNet
  | z_tf (latent TF activity)
  | x E1 (TF->region)
  | region_activity
  |---> ATAC reconstruction
  | x E2 (region->gene)
  +---> RNA reconstruction
```

## Training Stages

deepSCENIC supports three training stages:

1. **Pretraining (optional)**: Train TF2rNet + Enformer on DNA sequences before joint training
2. **Training**: Joint training of VAE + TF2rNet with warmup phases
3. **Finetuning (optional)**: Fine-tune only E2 matrix for transfer to new datasets

## Configure Training

In [8]:
# Training configuration
config = {
    "epochs": 100,
    "batch_size": 64,
    "seq_batch_size": 1000,
    "lr": 1e-4,
    "warmup_vae": 10,      # Epochs before updating TF2rNet
    "warmup_grn": 50,      # Epochs before enabling PPI
    "beta": 1e-2,          # KL loss weight
    "alpha": 1e-2,         # E1 sparsity weight
    "gamma": 1.0,          # E2 sparsity weight
}

## Stage 1: Pretraining (Optional)

Pretraining trains the sequence models (Enformer + TF2rNet) on DNA sequences
before joint training. This can help the model learn better TF binding patterns.

In [10]:
# Optional: Pretrain sequence models
pretrained = ds.tl.pretrain(
    mdata,
    epochs=1,
    batch_size=8,
    lr=1e-4,
    device=device,
)
pretrained.save("pretrained.pt")

KeyError: 'tfs'

## Stage 2: Train the Model

In [9]:
# Train deepSCENIC
model = ds.tl.train(
    mdata,
    epochs=config["epochs"],
    batch_size=config["batch_size"],
    seq_batch_size=config["seq_batch_size"],
    lr=config["lr"],
    warmup_vae=config["warmup_vae"],
    warmup_grn=config["warmup_grn"],
    # pretrained_model=pretrained,  # Use pretrained model if available
    device=device,
)

KeyError: 'tfs'

You can also use a TrainingConfig for more control:

In [ ]:
# Advanced configuration
# from deepscenic.tl import TrainingConfig
# config = TrainingConfig(
#     epochs=1000,
#     batch_size=64,
#     warmup_vae=10,
#     warmup_grn=50,
#     beta=0.01,
#     alpha=0.01,
#     gamma=1.0,
# )
# model = ds.tl.train(mdata, config=config, device=device)

## Stage 3: Finetuning (Optional)

Finetuning trains only the E2 matrix (region->gene) while keeping all other
parameters frozen. This is useful for adapting a trained model to new data.

In [ ]:
# Optional: Fine-tune E2 matrix
# model = ds.tl.finetune(
#     model,
#     mdata,
#     epochs=10000,
#     lr=1e-6,  # Very small learning rate
#     device=device,
# )

Key finetuning parameters:
- `epochs`: Typically 10000 (much higher than training)
- `lr`: Very small (1e-6), 100x lower than training
- `reinit_e2`: If True, reinitialize E2 to near-zero (default)
- `lr_patience`: Epochs before reducing LR (uses ReduceLROnPlateau)

## Monitor Training

Visualize training progress with loss curves:

In [11]:
# Plot loss curves
ds.pl.loss_curves(model)

NameError: name 'model' is not defined

## Evaluate Model Quality

Check the learned sparsity and latent representations:

In [ ]:
# Sparsity histogram of E1 and E2
# ds.pl.sparsity_histogram(model)

In [ ]:
# UMAP of latent TF activity space
# ds.pl.latent_umap(model, mdata, color="cell_type")

## Save the Model

In [ ]:
# Save the trained model
# model.save("deepscenic_model.pt")

## Load a Saved Model

In [ ]:
# Load a previously trained model
# model = ds.tl.load_model("deepscenic_model.pt")

# Load a pretrained model (sequence models only)
# pretrained = ds.tl.load_pretrained("pretrained.pt")

## Complete Workflow Example

Here is a complete workflow with all three stages:

In [ ]:
# 1. Pretrain sequence models (optional but recommended for large datasets)
# pretrained = ds.tl.pretrain(mdata, epochs=100, device=device)

# 2. Main training
# model = ds.tl.train(
#     mdata,
#     epochs=1000,
#     lr=1e-4,
#     pretrained_model=pretrained,  # Use pretrained weights
#     device=device,
# )

# 3. Fine-tune E2 for transfer learning (optional)
# model = ds.tl.finetune(model, mdata, epochs=10000, lr=1e-6, device=device)

# 4. Save the final model
# model.save("final_model.pt")

## Next Steps

In the next tutorial, we will extract and analyze the GRN:

- {doc}`03_grn_analysis`